# Sign Language Recognition with Deep Learning
Recognize hand gestures from **images** and **video** and convert them to **text**.



In [ ]:
# %pip install -q tensorflow opencv-python kagglehub scikit-learn matplotlib seaborn

In [ ]:
import os, json, glob, random, time
from pathlib import Path
import numpy as np
import cv2
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models, callbacks
from sklearn.metrics import classification_report, confusion_matrix

SEED = 42
random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)
print("TensorFlow:", tf.__version__)
print("GPUs:", tf.config.list_physical_devices("GPU"))

## Part A: CNN on the ASL Alphabet (image → letter)

### A1. Download the dataset
[`grassknoted/asl-alphabet`](https://www.kaggle.com/datasets/grassknoted/asl-alphabet): about 87,000 images of 200×200 px, 29 classes.

In [ ]:
import kagglehub
asl_root = kagglehub.dataset_download("grassknoted/asl-alphabet")
print("Downloaded to:", asl_root)

# Folder layout differs slightly between versions, so locate the train folder automatically
train_dir = None
for p in Path(asl_root).rglob("asl_alphabet_train"):
    subdirs = [d for d in p.iterdir() if d.is_dir()]
    if len(subdirs) >= 26:
        train_dir = p; break
    inner = p / "asl_alphabet_train"
    if inner.exists():
        train_dir = inner; break
print("Train dir:", train_dir)
print("Classes:", sorted(d.name for d in Path(train_dir).iterdir() if d.is_dir()))

### A2. Build `tf.data` pipelines (train / val / test)

In [ ]:
IMG_SIZE = 64        # 64x64 trains fast; raise to 96/128 for more accuracy
BATCH    = 64

train_ds = keras.utils.image_dataset_from_directory(
    train_dir, validation_split=0.2, subset="training", seed=SEED,
    image_size=(IMG_SIZE, IMG_SIZE), batch_size=BATCH, label_mode="int")
val_full = keras.utils.image_dataset_from_directory(
    train_dir, validation_split=0.2, subset="validation", seed=SEED,
    image_size=(IMG_SIZE, IMG_SIZE), batch_size=BATCH, label_mode="int")

class_names = train_ds.class_names
NUM_CLASSES = len(class_names)

# Split the held-out 20% in half: validation (during training) and test (final report)
n_val_batches = tf.data.experimental.cardinality(val_full).numpy()
val_ds  = val_full.take(n_val_batches // 2)
test_ds = val_full.skip(n_val_batches // 2)

AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.cache().shuffle(1000).prefetch(AUTOTUNE)
val_ds   = val_ds.cache().prefetch(AUTOTUNE)
test_ds  = test_ds.cache().prefetch(AUTOTUNE)
print(NUM_CLASSES, "classes")

In [ ]:
# Peek at a few samples
plt.figure(figsize=(10, 6))
for images, labels in train_ds.take(1):
    for i in range(12):
        plt.subplot(3, 4, i + 1)
        plt.imshow(images[i].numpy().astype("uint8"))
        plt.title(class_names[labels[i]]); plt.axis("off")
plt.tight_layout(); plt.show()